# G6: Điền khuyết đoạn PPG mất 2 giây trên PPG-DaLiA

Trịnh Nhật Anh, 23110074. Notebook này chỉ gọi các script trong thư mục `G6`, không chứa logic riêng. Xem `README.md` để biết thứ tự đầy đủ.

Cần upload lên Drive: thư mục `G6` (mã nguồn) và thư mục `dalia_cache` (45 tệp npz) nằm cạnh nhau, ví dụ `MyDrive/G6_PROJECT/G6` và `MyDrive/G6_PROJECT/dalia_cache`.

Mỗi lượt huấn luyện ghi checkpoint sau từng epoch và `results/<id>.json` khi xong; nếu Colab ngắt, chạy lại đúng ô đó là tiếp tục.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/G6_PROJECT/G6
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, numpy, scipy, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| numpy', numpy.__version__, '| scipy', scipy.__version__, '| cuda', torch.cuda.is_available())

## 0. Kiểm thử bắt buộc (mục 5.3), thống kê dữ liệu, bộ đọc HR, đối chuẩn trên validation

In [ ]:
!python -u tests_leak.py
!python -u data.py --thongke
!python -u hr_reader.py --chon
!python -u evaluate.py --doichuan --tap val

## 1. E4: khảo sát λ trên validation (chia cố định S1-10 / S11-12). Không nhìn test.

In [ ]:
!python -u experiments.py e4

## 2. E2: bóc tách thành phần. (a) bỏ Bi-LSTM, (c) thêm ACC, dùng {0, λ tốt nhất của E4}; (b) λ=0 dùng lại từ E4.

In [ ]:
!python -u experiments.py e2

## 3. KHÓA cấu hình. Sau ô này không đổi siêu tham số nữa (mục 5.1).

In [ ]:
!python -u experiments.py khoa
!cat results/khoa_cau_hinh.json

## 4. E1 và E3: đánh giá trên test S13-S15, chỉ chạy sau khi đã khóa.

In [ ]:
!python -u experiments.py e1

## 5. E5: kiểm định chéo 5 fold theo đối tượng (mục 2.4)
Đủ lưới λ {0; 0,1; 0,5; 1} ở mỗi fold (20 lượt), λ chọn trong validation riêng của từng fold. Sau đó đánh giá ngoài mẫu, bộ đọc HR chọn lại theo train/validation của từng fold.

In [ ]:
!python -u experiments.py e5 --luoi 0 0.1 0.5 1
!python -u evaluate.py --e5

## 6. Tài nguyên và độ trễ (mục 4.5, đo trên CPU khi máy rảnh), sinh hình, xuất bảng

In [ ]:
!python -u bench.py
!python -u figures.py
!python -u xuat_bang.py